# SAID — Kaggle P100 Training
Two-stage fine-tuning: VOC-FOG → RTTS

In [ ]:
# Cell 1: Upload datasets before running
# Kaggle sidebar → + Add Data → 'New Dataset'
# Upload zipped RTTS_Ready → name: rtts-ready
# Upload zipped VOC_FOG_YOLO → name: voc-fog-yolo
import os
print(os.listdir('/kaggle/input/'))

In [ ]:
# Cell 2: Install deps
!pip install -q ultralytics

In [ ]:
# Cell 3: Write said/__init__.py
import os; os.makedirs('said', exist_ok=True)
open('said/__init__.py','w').write('')
print('done')

In [ ]:
# Cell 4: Write a2c2f_fsa.py
code = open('/kaggle/input/said-code/said/a2c2f_fsa.py').read()
open('said/a2c2f_fsa.py','w').write(code)
print('a2c2f_fsa.py written')

In [ ]:
# Cell 5: Write wiou_loss.py
code = open('/kaggle/input/said-code/said/wiou_loss.py').read()
open('said/wiou_loss.py','w').write(code)
print('wiou_loss.py written')

In [ ]:
# Cell 6: Write YAML configs
rtts_yaml = '''path: /kaggle/input/rtts-ready
train: images/train
val: images/val
test: images/test
names:
  0: person
  1: bicycle
  2: car
  3: bus
  4: motorbike
'''
vocfog_yaml = '''path: /kaggle/input/voc-fog-yolo
train: images/train
val: images/val
test: images/test
names:
  0: person
  1: bicycle
  2: car
  3: bus
  4: motorbike
'''
open('rtts.yaml','w').write(rtts_yaml)
open('vocfog.yaml','w').write(vocfog_yaml)
print('YAMLs written')

In [ ]:
# Cell 7: Stage 1 — Pre-train on VOC-FOG (50 epochs)
from ultralytics import YOLO
model = YOLO('yolov12x.pt')
results = model.train(
    data='vocfog.yaml', epochs=50, batch=32, imgsz=640,
    device='0', project='runs/said', name='stage1_vocfog',
    optimizer='AdamW', lr0=0.01, weight_decay=0.0005,
    mosaic=1.0, mixup=0.15, copy_paste=0.1,
    close_mosaic=10, patience=15, save=True, plots=True
)
import shutil, pathlib
best = pathlib.Path(results.save_dir)/'weights'/'best.pt'
shutil.copy(best, 'said_stage1.pt')
print('Stage 1 done →', best)

In [ ]:
# Cell 8: Stage 2a — Freeze backbone 10 epochs
from ultralytics import YOLO
model = YOLO('said_stage1.pt')
model.train(
    data='rtts.yaml', epochs=10, batch=32, imgsz=640,
    device='0', project='runs/said', name='stage2a_freeze',
    freeze=list(range(10)), lr0=0.001, optimizer='AdamW',
    mosaic=0.8, val=True, save=True
)
print('Stage 2a done')

In [ ]:
# Cell 9: Stage 2b — Full fine-tune with eval schedule
import csv, json, shutil
from pathlib import Path
from ultralytics import YOLO

RUN_DIR  = Path('runs/said/stage2b_full')
LOG_PATH = RUN_DIR / 'eval_log.csv'

state = {'best_map50': 0.0, 'best_epoch': -1}

def _val(tag, epoch, split, yaml):
    last = RUN_DIR/'weights'/'last.pt'
    m = YOLO(str(last)) if last.exists() else model
    mt = m.val(data=yaml, imgsz=640, batch=32, device='0',
                split=split, plots=False, verbose=False)
    row = {'epoch':epoch,'tag':tag,'split':split,
           'map50':round(mt.box.map50,4),'map5095':round(mt.box.map,4)}
    write_hdr = not LOG_PATH.exists()
    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(LOG_PATH,'a',newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        if write_hdr: w.writeheader()
        w.writerow(row)
    print(f'  [{tag}|ep{epoch}] mAP50={row["map50"]}')
    return row

def on_fit_epoch_end(trainer):
    ep = trainer.epoch + 1
    if ep % 10 == 0:
        row = _val('VAL-RTTS', ep, 'val', 'rtts.yaml')
        if ep % 5 == 0 and row['map50'] > state['best_map50']:
            state['best_map50'] = row['map50']
            state['best_epoch'] = ep
            last = RUN_DIR/'weights'/'last.pt'
            if last.exists(): shutil.copy2(last,'said_rolling_best.pt')
            print(f'  ★ New best mAP50={row["map50"]} @ ep{ep}')
    if ep % 20 == 0:
        _val('TEST-RTTS',   ep, 'test', 'rtts.yaml')
        _val('TEST-VOCFOG', ep, 'test', 'vocfog.yaml')

def on_train_end(trainer):
    ep = trainer.epoch + 1
    _val('FINAL-VAL',  ep, 'val',  'rtts.yaml')
    _val('FINAL-TEST', ep, 'test', 'rtts.yaml')
    best = RUN_DIR/'weights'/'best.pt'
    if best.exists(): shutil.copy2(best, 'said_final.pt')
    last = RUN_DIR/'weights'/'last.pt'
    if last.exists(): shutil.copy2(last, 'said_last.pt')
    print(f'Best rolling mAP50={state["best_map50"]} @ ep{state["best_epoch"]}')
    print(f'Eval log: {LOG_PATH}')

phase2a_best = Path('runs/said/stage2a_freeze/weights/best.pt')
model = YOLO(str(phase2a_best))
model.add_callback('on_fit_epoch_end', on_fit_epoch_end)
model.add_callback('on_train_end', on_train_end)
model.train(
    data='rtts.yaml', epochs=100, batch=32, imgsz=640,
    device='0', project='runs/said', name='stage2b_full',
    optimizer='AdamW', lr0=0.01, weight_decay=0.0005,
    mosaic=1.0, mixup=0.15, copy_paste=0.1,
    close_mosaic=10, patience=0, val=False, save=True, plots=True
)
print('Training complete')

In [ ]:
# Cell 10: Final results
from ultralytics import YOLO
model = YOLO('said_final.pt')
for split in ['val','test']:
    m = model.val(data='rtts.yaml', split=split, imgsz=640, batch=32, device='0', plots=True)
    print(f'RTTS {split} — mAP50: {m.box.map50:.4f}  mAP50:95: {m.box.map:.4f}')